In [ ]:
import pandas as pd
import sqlite3

In [ ]:
df_fact_orders= pd.read_csv('fact_orders.csv')
df_fact_ratings = pd.read_csv('fact_ratings.csv')
df_fact_order_items = pd.read_csv('fact_order_items.csv')
df_fact_delivery_performance = pd.read_csv('fact_delivery_performance.csv')
df_dim_restaurant= pd.read_csv('dim_restaurant.csv')
df_dim_menu_item = pd.read_csv('dim_menu_item.csv')
df_dim_delivery_partner = pd.read_csv('dim_delivery_partner_.csv')
df_dim_customer = pd.read_csv('dim_customer.csv')

In [ ]:
conn = sqlite3.connect('example.db')

In [ ]:
df_fact_orders.to_sql('fact_orders',conn)
df_fact_ratings.to_sql('fact_ratings',conn)
df_fact_order_items.to_sql('fact_order_items',conn)
df_fact_delivery_performance.to_sql('fact_delivery_performance',conn)
df_dim_restaurant.to_sql('dim_restaurant',conn)
df_dim_menu_item.to_sql('dim_menu_item',conn)
df_dim_delivery_partner.to_sql('dim_delivery_partner',conn)
df_dim_customer.to_sql('dim_customer',conn)

107776

In [ ]:
%load_ext sql

The sql extension is already loaded. To reload it, use:
  %reload_ext sql


In [ ]:
%sql sqlite:///example.db

### 1. Compare total orders across pre-crisis (Jan–May 2025) vs crisis (Jun–Sep 2025).

In [ ]:
%%sql order_results <<
WITH order_counts AS (
	SELECT SUM(CASE WHEN DATE(order_timestamp) BETWEEN '2025-01-01' AND '2025-05-01' THEN 1 ELSE 0 END) AS pre_crisis,
		   SUM(CASE WHEN DATE(order_timestamp) BETWEEN '2025-06-01' AND '2025-10-01' THEN 1 ELSE 0 END) AS post_crisis
	FROM fact_orders
    WHERE is_cancelled = 'N'
)
SELECT (pre_crisis-post_crisis) AS total_drop ,ROUND((pre_crisis-post_crisis)*100/pre_crisis,2) AS declined_precentage
FROM order_counts ;

Running query in 'sqlite:///example.db'

In [ ]:
order_results.DataFrame()

,total_drop,declined_precentage
0,55301,63.0


 ### 2. Which top 5 city groups experienced the highest percentage decline in orders during the crisis period compared to the pre-crisis period.


In [ ]:
%%sql r2 <<

WITH orders_city AS (
	SELECT t1.order_id,t1.order_timestamp,t2.city,t1.is_cancelled,t2.is_active FROM fact_orders t1
	JOIN dim_restaurant t2 ON t1.restaurant_id = t2.restaurant_id
), city_declined_orders AS (
SELECT city,SUM(CASE WHEN DATE(order_timestamp) BETWEEN '2025-01-01' AND '2025-05-31' THEN 1 ELSE 0 END) AS pre_crisis ,
		   SUM(CASE WHEN DATE(order_timestamp) BETWEEN '2025-06-01' AND '2025-10-30' THEN 1 ELSE 0 END)  AS post_crisis
FROM orders_city
WHERE is_cancelled = 'N'
GROUP BY city)

SELECT city FROM (
SELECT city, RANK() OVER(ORDER BY (pre_crisis - post_crisis) * 100 / pre_crisis DESC)  AS city_declined_rank
FROM city_declined_orders) t1
WHERE city_declined_ranK <=5;


Running query in 'sqlite:///example.db'

In [ ]:
r2.DataFrame()

,city
0,Ahmedabad
1,Bengaluru
2,Chennai
3,Kolkata
4,Delhi
5,Hyderabad
6,Mumbai


### 3. Among restaurants with at least 50 pre-crisis orders, which top 10 high-volume restaurants experienced the largest percentage decline in order counts during the crisis period?


In [ ]:
%%sql r3 <<
WITH restaurant_orders AS (
SELECT t1.order_id,t1.order_timestamp,t1.is_cancelled,t2.restaurant_id,t2.restaurant_name FROM fact_orders t1
JOIN dim_restaurant t2 ON t1.restaurant_id = t2.restaurant_id
WHERE is_cancelled = 'N'),
restaurant_declined_orders AS (
SELECT restaurant_name,SUM(CASE WHEN DATE(order_timestamp) BETWEEN '2025-01-01' AND '2025-05-31' THEN 1 ELSE 0 END) AS pre_crisis ,
		   SUM(CASE WHEN DATE(order_timestamp) BETWEEN '2025-06-01' AND '2025-10-30' THEN 1 ELSE 0 END)  AS post_crisis
FROM restaurant_orders
GROUP BY restaurant_name
)


SELECT restaurant_name FROM (
SELECT restaurant_name,
	   pre_crisis,
       post_crisis, ROUND((pre_crisis-post_crisis)*100/pre_crisis) AS declined_percentage FROM restaurant_declined_orders
       WHERE pre_crisis >=50
       )T3

ORDER BY declined_percentage DESC
LIMIT 10;


Running query in 'sqlite:///example.db'

In [ ]:
r3.DataFrame()

,restaurant_name
0,Fresh Tandoor Delight
1,Urban Kitchen Zone
2,Classic Sweets Heaven
3,Flavours of Tandoor Central
4,Grand Cafe Clouds
5,Hot & Crispy Mess Mahal
6,Punjabi Curry Delight
7,Punjabi Sweets Cafe
8,Thindi Mane Darshini Heaven
9,Delhi Tandoor Mahal


### 4. What is the cancellation rate trend pre-crisis vs crisis,and which cities are most affected?


In [ ]:
%%sql r4 <<
WITH orders_city AS (
	SELECT t1.order_id,t1.order_timestamp,t2.city,t1.is_cancelled,t2.is_active FROM fact_orders t1
	JOIN dim_restaurant t2 ON t1.restaurant_id = t2.restaurant_id
), orders_city_cancelled AS (
SELECT * FROM orders_city
WHERE is_cancelled = 'Y'
),  city_cancelled_orders AS (
SELECT city,SUM(CASE WHEN DATE(order_timestamp) BETWEEN '2025-01-01' AND '2025-5-31' THEN 1 ELSE 0 END)  AS precrisis_cancelled_orders,
		   SUM(CASE WHEN DATE(order_timestamp) BETWEEN '2025-06-01' AND '2025-10-30' THEN 1 ELSE 0 END)  AS crisis_cancelled_orders
FROM orders_city_cancelled
GROUP BY city
),city_total_orders AS
(
SELECT city,SUM(CASE WHEN DATE(order_timestamp) BETWEEN '2025-01-01' AND '2025-5-31' THEN 1 ELSE 0 END)  AS total_orders_precrisis,
		   SUM(CASE WHEN DATE(order_timestamp) BETWEEN '2025-06-01' AND '2025-10-30' THEN 1 ELSE 0 END)  AS total_orders_crisis  FROM orders_city
GROUP BY city
),combined AS (
SELECT t1.city,total_orders_precrisis,total_orders_crisis,precrisis_cancelled_orders,crisis_cancelled_orders FROM city_cancelled_orders t1
LEFT JOIN city_total_orders t2 ON t1.city = t2.city
)

SELECT city,(ROUND(precrisis_cancelled_orders*100/total_orders_precrisis,2) - ROUND(crisis_cancelled_orders*100/total_orders_crisis,2)) AS rate_spike FROM combined
ORDER BY rate_spike ;

Running query in 'sqlite:///example.db'

In [ ]:
r4.DataFrame()

,city,rate_spike
0,Ahmedabad,-6.0
1,Chennai,-5.0
2,Hyderabad,-5.0
3,Mumbai,-5.0
4,Bengaluru,-4.0
5,Delhi,-4.0
6,Kolkata,-4.0
7,Pune,-4.0


### 5.  Measure average delivery time across phases. Did SLA compliance worsen significantly in the crisis period?


In [ ]:
%%sql r5 <<
WITH delivery_across_phases AS(
SELECT actual_delivery_time_mins,expected_delivery_time_mins,
CASE WHEN DATE(order_timestamp) BETWEEN '2025-01-01' AND '2025-05-31' THEN 'pre_crisis'
			  WHEN DATE(order_timestamp) BETWEEN '2025-06-01' AND '2025-10-30' THEN 'crisis'
              END
				AS phase
 FROM fact_delivery_performance t1
JOIN fact_orders t2 ON t1.order_id = t2.order_id
WHERE is_cancelled  = 'N'
),avg_delivery_across_phases AS (
SELECT phase,AVG(actual_delivery_time_mins) FROM delivery_across_phases
GROUP BY phase
)

SELECT phase,SUM(CASE WHEN actual_delivery_time_mins<=expected_delivery_time_mins THEN 1 END) AS on_time_delivery,
			COUNT(*) AS total_deliveries,
			(SUM(CASE WHEN actual_delivery_time_mins>expected_delivery_time_mins THEN 1 END)*100) /COUNT(*) AS sla_compliance_pct

FROM delivery_across_phases
GROUP BY phase;


Running query in 'sqlite:///example.db'

In [ ]:
r5.DataFrame()

,phase,on_time_delivery,total_deliveries,sla_compliance_pct
0,crisis,3828,31142,87
1,pre_crisis,46623,106912,56


### 6. Track average customer rating month-by-month. Which months saw the sharpest drop?

In [ ]:
%%sql r6 <<
WITH rating_month_cte AS (
    SELECT rating_month, AVG(rating) AS avg_rating
    FROM (
        SELECT *,
               -- Slices out 'YYYY' and 'MM' and joins them with a hyphen
               substr(review_timestamp, 7, 4) || '-' || substr(review_timestamp, 1, 2) AS rating_month
        FROM fact_ratings
    ) t1
    WHERE rating_month IS NOT NULL
    GROUP BY rating_month
    ORDER BY rating_month
)
SELECT rating_month,
       ROUND((avg_rating - previous_month_rating) * 100, 2) AS change_in_rating
FROM (
    SELECT *,
           LAG(avg_rating) OVER(ORDER BY rating_month) AS previous_month_rating
    FROM rating_month_cte
) t2
ORDER BY change_in_rating;

Running query in 'sqlite:///example.db'

In [ ]:
r6.DataFrame()

,rating_month,change_in_rating
0,2025-01,NaN
1,2025-29,-6.07
2,2025-19,-5.99
3,2025-28,-5.01
4,2025-05,-4.67
5,2025-14,-4.57
6,2025-02,-4.21
7,2025-25,-3.96
8,2025-07,-3.47
9,2025-09,-2.90


### 7. Estimate revenue loss from pre-crisis vs crisis (based on subtotal, discount, and delivery fee).


In [ ]:
%%sql r7 <<
WITH revenue AS (
SELECT SUM(CASE WHEN DATE(order_timestamp) BETWEEN '2025-01-01' AND '2025-05-31' THEN total_amount END ) AS pre_crisis_revenue,
      SUM(CASE WHEN DATE(order_timestamp) BETWEEN '2025-06-01' AND '2025-10-30' THEN total_amount END )AS crisis_revenue FROM fact_orders)


SELECT ROUND((pre_crisis_revenue-crisis_revenue)*100/pre_crisis_revenue,2) AS loss_pct FROM revenue;



Running query in 'sqlite:///example.db'

In [ ]:
r7.DataFrame()

,loss_pct
0,70.92


### 8. : Among customers who placed five or more orders before the crisis, determine how many stopped ordering during the crisis, and out of those, how many had an average rating above 4.5?


In [ ]:
%%sql r8 <<
WITH customer_orders AS (

SELECT t1.customer_id,order_timestamp,is_cancelled
FROM dim_customer t1
JOIN fact_orders t2 ON t1.customer_id = t2.customer_id

), precrisis_frequent_customers AS (
SELECT customer_id
FROM  customer_orders
WHERE DATE(order_timestamp) < '2025-06-01'
GROUP BY customer_id
HAVING COUNT(*) >=5
),
crisis_frequent_customers AS (
SELECT customer_id  FROM customer_orders
WHERE DATE(order_timestamp)>='2025-06-01' AND customer_id IN (SELECT customer_id FROM precrisis_frequent_customers)
),lost_loylty_customers AS (
SELECT t1.customer_id FROM precrisis_frequent_customers t1
LEFT JOIN crisis_frequent_customers t2 ON t1.customer_id = t2.customer_id
WHERE t2.customer_id IS NULL
)

SELECT t1.customer_id,AVG(rating) FROM lost_loylty_customers t1
JOIN fact_ratings t2 ON t1.customer_id = t2.customer_id
GROUP BY t1.customer_id
HAVING AVG(rating)>4.5;



Running query in 'sqlite:///example.db'

In [ ]:
r8.DataFrame()

,customer_id,AVG(rating)
0,CUST026722,4.566667
1,CUST032044,4.850000
2,CUST032334,5.000000
3,CUST041953,5.000000
4,CUST042658,4.733333
5,CUST061759,4.750000
6,CUST069956,4.550000
7,CUST078309,4.750000
8,CUST082992,4.700000
9,CUST083875,5.000000


### 9. Which high-value customers (top 5% by total spend before the crisis) showed the largest drop in order frequency and ratings during the crisis? What common patterns (e.g., location, cuisine preference, delivery delays) do they share?



In [ ]:
%%sql r9 <<

WITH customer_spend_pre_crisis AS(
SELECT t1.customer_id,SUM(t1.total_amount) AS total_spend_pre,COUNT(t1.order_id) AS order_freq_pre,AVG(rating) AS avg_rating_pre FROM fact_orders t1
JOIN fact_ratings t2 ON t1.order_id = t2.order_id
WHERE is_cancelled  = 'N' AND
DATE(order_timestamp) <'2025-05-01'
GROUP BY t1.customer_id
ORDER BY total_spend_pre DESC
),ranked_customers AS (
	SELECT customer_id,order_freq_pre,avg_rating_pre,total_spend_pre,PERCENT_RANK() OVER(ORDER BY total_spend_pre DESC) AS spending_percentile
    FROM customer_spend_pre_crisis
),top_spenders AS (
SELECT
    customer_id,order_freq_pre,avg_rating_pre,
    total_spend_pre
FROM ranked_customers
WHERE spending_percentile <= 0.05
ORDER BY total_spend_pre DESC),
customer_spend_crisis AS(

SELECT t1.customer_id,SUM(t1.total_amount) AS total_spend_crisis,COUNT(t1.order_id) AS order_freq_crisis,AVG(rating) AS avg_rating_crisis FROM fact_orders t1
JOIN fact_ratings t2 ON t1.order_id = t2.order_id
WHERE is_cancelled  = 'N' AND
DATE(order_timestamp) >='2025-06-01' AND t1.customer_id IN (SELECT customer_id FROM top_spenders)
GROUP BY t1.customer_id
ORDER BY total_spend_crisis DESC

)

SELECT t1.customer_id,ROUND((t1.order_freq_pre-COALESCE(t2.order_freq_crisis,0))*100/t1.order_freq_pre,2) AS freq_drop_pct,
ROUND((avg_rating_pre-avg_rating_crisis)*100/avg_rating_pre,2) AS rating_drop_pct FROM top_spenders t1
LEFT JOIN customer_spend_crisis t2 ON t1.customer_id = t2.customer_id
ORDER BY freq_drop_pct;







Running query in 'sqlite:///example.db'

In [ ]:
r9.DataFrame()

,customer_id,freq_drop_pct,rating_drop_pct
0,CUST060446,-100.0,43.33
1,CUST159977,0.0,58.70
2,CUST021364,0.0,48.39
3,CUST117257,0.0,58.95
4,CUST018924,0.0,45.65
...,...,...,...
1917,CUST161461,100.0,NaN
1918,CUST188922,100.0,NaN
1919,CUST136823,100.0,NaN
1920,CUST119969,100.0,NaN
